# Categorization model bake-off runbook

A step-by-step playbook to **redo the poem-categorization model bake-off from scratch**.
Open it, run the cells in order, and you end up with a gold reference set, a
scored comparison of candidate classifier models, and a recommendation for which
model to use for the full 9k-poem bulk run.

## What a bake-off is here

The categorization pipeline tags every poem with a controlled taxonomy: moods,
topics, motifs (multi-label) plus three scalars (`mood_primary`,
`emotional_intensity`, `accessibility_level`). Different LLMs do this with
different quality, cost, and speed. A bake-off:

1. Builds a small **gold set**: a stratified sample of poems labeled by a strong,
   expensive **judge** model (`gemini-3.1-pro-preview`), spot-checked against a
   human.
2. Runs each cheap **candidate** model over the same poems.
3. Scores every candidate against the gold on multi-label F1 / Jaccard,
   `mood_primary` accuracy, and scalar MAE, sitting cost ($/1k) and throughput
   (poems/min) right next to quality.
4. Prints a recommendation and writes an HTML chart.

## When to re-run it

- **New models available.** A newer/cheaper Flash tier ships; see if it beats the
  incumbent.
- **Taxonomy change.** The vocab in `config.py` changed (`TAXONOMY_VERSION` bump),
  so old labels no longer compare, so rebuild gold and re-bake.
- **Provider/pricing shift.** Costs moved enough to change the quality-vs-cost call.

## The harness

Everything runs through one module with four mutually-exclusive modes:

```
python -m poetry_quality_and_curation.categorization.eval_categorization \
    ( --build-gold | --emit-review | --apply-corrections FILE | --bakeoff )
    [--judge M] [--candidates M ...] [--sample-size N] [--pool N]
    [--seed N] [--batch-size N] [--concurrency N]
```

The scoring / sampling / HTML functions take plain data (poems + labels), so they
are unit-testable with no DB and no network. Only `--build-gold`, `--bakeoff`, and
the labeling steps touch the API.

> **Read `poetry_quality_and_curation/categorization/eval_categorization.py`
> alongside this notebook.** The module docstring and `parse_args()` are the
> source of truth for flags; this runbook mirrors them but the code wins.

## ⚠️ Read this before you run anything: the two kinds of 404

The Gemini endpoint returns **HTTP 404 in two completely different situations**.
Confusing them will waste hours, so learn to tell them apart:

| Symptom | Meaning | Fix |
|---|---|---|
| **404 with an *empty* body**, on rapid back-to-back calls | You are being **rate-limited by bursts**. Gemini does *not* always return the polite 429 here, so a burst trips an empty-body 404. | **Slow down.** Run at `--concurrency 1` (each call self-paces on its own latency, roughly 5 to 7s apart) or add backoff. Paced calls give ~100% success. |
| **404 *with* a body** saying the model "is no longer available" / "not found" | The **model is genuinely retired** by Google. | Pick a different model. Nothing you do to pacing helps. |

**Practical rule for this runbook: run every API-hitting step at
`CONCURRENCY = 1`.** It is slower but reliable. The gold build is only ~120
poems and the bake-off is 120 by N candidates, so a few extra minutes is nothing
next to a half-finished run that tripped the limiter. Only raise concurrency once
you have added real exponential backoff around the LiteLLM call.

If a whole batch fails, the harness prints `[warn] batch failed on <model>: ...`
and keeps going (that poem is simply absent from the scored set), so a transient
404 degrades coverage rather than crashing the run. It also silently shrinks
your gold, though, so watch the printed poem counts.

## Parameters

This is the **one cell to edit** when re-running with different settings. Every
later code cell reads from these variables, so changing a model or sample size
here propagates everywhere. Run this cell first (it hits no network).

In [ ]:
from pathlib import Path
import shlex, sys

# --- Repo location -----------------------------------------------------------
# This notebook lives in poetry_quality_and_curation/categorization/. The repo
# root is a few levels up. Adjust if you moved the notebook.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
for _ in range(6):
    if (REPO_ROOT / "poetry_quality_and_curation").is_dir() and (REPO_ROOT / ".git").exists():
        break
    REPO_ROOT = REPO_ROOT.parent
print("REPO_ROOT =", REPO_ROOT)

# --- Python / venv -----------------------------------------------------------
# The module CLIs run in a venv that has requirements.txt installed. Point PY at
# that interpreter. Default: whatever is running this notebook.
PY = sys.executable   # e.g. ".../categorization/.venv/bin/python"

# --- Model roster ------------------------------------------------------------
JUDGE       = "gemini/gemini-3.1-pro-preview"          # strong reference labeler
CANDIDATES  = [
    "gemini/gemini-2.5-flash",
    "gemini/gemini-3.5-flash",
    "gemini/gemini-3.6-flash",
]
# NOTE: "gemini/gemini-3-pro-preview" is RETIRED (404-with-body). Do not add it.

# --- Sampling / gold ---------------------------------------------------------
SAMPLE_SIZE = 120     # poems in the gold set (harness default)
POOL        = 4000    # DB rows to sample the stratified gold from
SEED        = 42      # deterministic sample for a given seed
BATCH_SIZE  = 4       # poems per LLM call

# --- Reliability -------------------------------------------------------------
# Keep this at 1. See the "two kinds of 404" section above.
CONCURRENCY = 1

# --- Data outputs (written by the harness) -----------------------------------
DATA_DIR      = REPO_ROOT / "poetry_quality_and_curation" / "categorization" / "data"
GOLD_PATH     = DATA_DIR / "categorization_gold.json"
REVIEW_PATH   = DATA_DIR / "categorization_gold_review.json"
BAKEOFF_JSON  = DATA_DIR / "categorization_bakeoff.json"
BAKEOFF_HTML  = DATA_DIR / "categorization_bakeoff.html"

MODULE = "poetry_quality_and_curation.categorization.eval_categorization"

def cli(*args):
    """Build a copy-pasteable module invocation string from the params above."""
    parts = [PY, "-m", MODULE, *[str(a) for a in args]]
    return " ".join(shlex.quote(p) for p in parts)

print("Judge      :", JUDGE)
print("Candidates :", CANDIDATES)
print(f"Sample={SAMPLE_SIZE}  Pool={POOL}  Seed={SEED}  Batch={BATCH_SIZE}  Concurrency={CONCURRENCY}")
print("Data dir   :", DATA_DIR)

## Prerequisites & environment

### 1. Virtualenv + dependencies

From the repo root:

```bash
python3 -m venv poetry_quality_and_curation/categorization/.venv
source poetry_quality_and_curation/categorization/.venv/bin/activate
pip install -r poetry_quality_and_curation/categorization/requirements.txt
```

`requirements.txt` pulls `litellm`, `psycopg2-binary`, `pandas`, `pyarrow`,
`tqdm`, `python-dotenv`. Set `PY` in the parameters cell to that venv's
`bin/python` so the cells below use it.

### 2. `.env` at the repo root

The pipeline calls `load_dotenv()` on import, so it reads a `.env` in the repo
root. It needs:

| Var | Purpose | Notes |
|---|---|---|
| `DATABASE_URL` | Supabase Postgres connection | **Pooler host, port 6543** (`aws-N-region.pooler.supabase.com:6543`), not the direct `db.*.supabase.co` host |
| `GEMINI_API_KEY` | Gemini auth for all `gemini/*` models | Routed with no `api_base` (see provider wiring below) |

Never print these values. The cells below only *check existence*, never echo.

### 3. Provider wiring (why the model strings matter)

`config.resolve_provider(model)` routes by prefix:

- `gemini/*` goes to Gemini directly via `GEMINI_API_KEY`, with **no `api_base`**.
- `openai/bedrock-*` goes to the Anthropic/LiteLLM proxy (`ANTHROPIC_BASE_URL` /
  `ANTHROPIC_AUTH_TOKEN`).

For Gemini 3.x, `classify_poems.py` disables thinking (`reasoning_effort`
handling). **This matters:** if thinking is left on, the model burns its token
budget "reasoning" and truncates the JSON, so parsing fails. That handling is
already in the classifier, so you don't need to do anything, but it is why the
Gemini 3.x models behave.

### Verify the environment (safe: no secrets printed, no API calls)

This cell confirms `.env` has the two required keys **without displaying their
values**, and checks the module imports.

In [ ]:
import os

env_path = REPO_ROOT / ".env"
print(".env present:", env_path.exists(), "->", env_path)

def has_key(name):
    # existence-only check; never prints the value
    if not env_path.exists():
        return False
    for line in env_path.read_text(encoding="utf-8").splitlines():
        if line.strip().startswith(f"{name}="):
            return bool(line.split("=", 1)[1].strip())
    return False

for key in ("DATABASE_URL", "GEMINI_API_KEY"):
    print(f"{key:16s}: {'configured' if has_key(key) else 'MISSING'}")

# Confirm the module is importable from this interpreter.
import subprocess
r = subprocess.run(
    [PY, "-c", "import poetry_quality_and_curation.categorization.eval_categorization as m; print('import OK')"],
    cwd=REPO_ROOT, capture_output=True, text=True,
)
print(r.stdout.strip() or r.stderr.strip())

### Verify DB connectivity (read-only `SELECT count(*)`)

A harmless read that confirms `DATABASE_URL` reaches the corpus. Expect roughly
**9,072 poems** in the current production DB. (Some older docs mention 84k, but
that number is stale; trust the count this prints.)

This does **not** call any LLM, so it is safe to run now.

In [ ]:
import subprocess, textwrap

snippet = textwrap.dedent("""
    from dotenv import load_dotenv; load_dotenv()
    from poetry_quality_and_curation.categorization import config
    conn = config.get_db_connection()
    try:
        cur = conn.cursor()
        cur.execute("SELECT count(*) FROM poems")
        print("poems in DB:", cur.fetchone()[0])
    finally:
        conn.close()
""")
r = subprocess.run([PY, "-c", snippet], cwd=REPO_ROOT, capture_output=True, text=True)
print(r.stdout.strip())
if r.returncode != 0:
    print("STDERR:", r.stderr.strip()[:800])

## Model roster

Validated models for this bake-off. The judge is the expensive, high-quality
reference; candidates are the cheap bulk options being compared.

| Role | Model string | Status | Notes |
|---|---|---|---|
| **Judge** | `gemini/gemini-3.1-pro-preview` | ✅ available | Labels the gold set; thinking allowed. |
| Candidate | `gemini/gemini-2.5-flash` | ✅ available | Cheapest; the current incumbent bulk model. |
| Candidate | `gemini/gemini-3.5-flash` | ✅ available | Higher quality, higher cost than 2.5. |
| Candidate | `gemini/gemini-3.6-flash` | ✅ available | Best quality of the three in the last run. |
| ~~Judge/candidate~~ | `gemini/gemini-3-pro-preview` | ❌ **RETIRED** | Returns **404 *with* a body** ("no longer available"). Do **not** use. |

All of these route through `GEMINI_API_KEY`. Anthropic/Bedrock strings
(`openai/bedrock-*`) also work via the proxy but were not part of this roster.

## Step A: Build the gold set (`--build-gold`)

**What it does.** Loads up to `POOL` poems from the DB (highest quality first),
draws a **stratified sample** of `SAMPLE_SIZE` across `(era_id, quality-band)`
strata (deterministic for `SEED`), then labels every sampled poem with the
**judge** model. It writes `data/categorization_gold.json` where each poem has:

- `pro`: the judge's labels (kept forever, even after human corrections).
- `human`: `null` for now, the slot a human fills in Step B.

**The command** (parameterized from the params cell):

In [ ]:
print(cli("--build-gold",
          "--judge", JUDGE,
          "--sample-size", SAMPLE_SIZE,
          "--pool", POOL,
          "--seed", SEED,
          "--batch-size", BATCH_SIZE,
          "--concurrency", CONCURRENCY))

**Run it** (uncomment the `!` line). This **hits the Gemini API**, so only run
when no other bake-off is live, at `CONCURRENCY = 1`.

**Expected output** (reference from the last run):

```
Loaded 4000 candidate poems from DB
Sampled 120 poems (stratified by era + quality band)
Labeling with judge gemini/gemini-3.1-pro-preview ...
  judge cost $0.68, 24 poems/min
Wrote gold set -> .../data/categorization_gold.json (120 poems)
```

So: **120 gold poems, ~$0.68, ~24 poems/min** with the 3.1-pro judge. At
`CONCURRENCY = 1` the throughput is latency-bound; expect it to take a few
minutes. If the poem count comes back well under 120, batches were tripping the
empty-body 404, so slow down further.

In [ ]:
# !{cli("--build-gold", "--judge", JUDGE, "--sample-size", SAMPLE_SIZE, \
#       "--pool", POOL, "--seed", SEED, "--batch-size", BATCH_SIZE, \
#       "--concurrency", CONCURRENCY)}

Inspect the gold set that was written (safe, no API):

In [ ]:
import json
if GOLD_PATH.exists():
    gold = json.loads(GOLD_PATH.read_text(encoding="utf-8"))
    poems = gold.get("poems", {})
    n_human = sum(1 for e in poems.values() if e.get("human"))
    print("judge_model     :", gold.get("judge_model"))
    print("taxonomy_version:", gold.get("taxonomy_version"))
    print("seed            :", gold.get("seed"))
    print("gold poems      :", len(poems))
    print("human-corrected :", n_human)
    # peek at one poem's judge labels
    first = next(iter(poems.values()))
    print("sample pro labels:", {k: first["pro"].get(k) for k in ("moods","topics","motifs","mood_primary")})
else:
    print("No gold file yet; run Step A first.")

## Step B: Human spot-check + calibration (`--emit-review` -> edit -> `--apply-corrections`)

This is the **hybrid** part of "hybrid gold set": a human validates a small slice
(the first 20 poems, `SPOT_CHECK_SIZE`) so you know whether to trust the judge on
the other 100.

### B1: Emit the review slice

Dumps the first 20 gold poems (content + the judge's `pro` labels + an empty
`human` slot) to `data/categorization_gold_review.json`. **No API call.**

In [ ]:
print(cli("--emit-review"))

In [ ]:
# Safe to run: this touches only local files (loads gold, writes the review JSON).
import subprocess
r = subprocess.run([PY, "-m", MODULE, "--emit-review"], cwd=REPO_ROOT,
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())

### B2: Edit the review file by hand

Open `data/categorization_gold_review.json`. For each poem, **copy the `pro`
block into `human` and correct the labels** (keys must come from the taxonomy
vocab in `config.py`). Leave `human` as `null` to accept the judge's labels
unchanged. This is a human judgment step, so there is no cell for it.

### B3: Apply corrections and read the calibration decision

`--apply-corrections` patches the human labels into the gold, then scores
**judge-vs-human agreement** on the corrected slice and decides whether a
recalibration is warranted:

- Agreement clears **both** thresholds (`mood_primary` accuracy >= 0.85 **and**
  mean multi-label Jaccard >= 0.70): **keep the judge's labels** for the other
  100 poems. The judge is trustworthy.
- Either threshold missed: **recalibration fires**. Re-label the full gold with
  the judge using the 20 human corrections as **few-shot** examples, then re-run
  `--apply-corrections`. This is the "escalating calibration": you only pay to
  re-label everything if the cheap 20-poem check says the judge drifted.

**No API call** in `--apply-corrections` itself (it only scores existing labels);
the *re-label* it may recommend is a separate judge run.

In [ ]:
print(cli("--apply-corrections", REVIEW_PATH))

Run it once you've edited the review file (safe: scores local data, no API).

**Expected output** shape:

```
Applied 20 human corrections
Judge-vs-human agreement over 20 poems: macro-Jaccard 0.7xx, mood_primary 0.8xx
Calibration decision: not fired (agreement OK), keeping judge labels
```

In [ ]:
# Uncomment after editing the review file. Local-only scoring, no Gemini call.
# import subprocess
# r = subprocess.run([PY, "-m", MODULE, "--apply-corrections", str(REVIEW_PATH)],
#                    cwd=REPO_ROOT, capture_output=True, text=True)
# print(r.stdout.strip() or r.stderr.strip())

## Step C: Run the bake-off (`--bakeoff`)

**What it does.** For each candidate model, labels all gold poems, then scores the
predictions against the **effective gold** (human labels where present, else the
judge's `pro`). Writes `data/categorization_bakeoff.json` and a self-contained
`data/categorization_bakeoff.html` chart, and prints a recommendation.

**The command:**

In [ ]:
print(cli("--bakeoff",
          "--candidates", *CANDIDATES,
          "--judge", JUDGE,
          "--batch-size", BATCH_SIZE,
          "--concurrency", CONCURRENCY))

**Run it** (uncomment). This **hits the API once per candidate over all 120 gold
poems**, the most expensive step. `CONCURRENCY = 1`, no live bake-off elsewhere.

### How to read the metrics

| Metric | What it measures | Better |
|---|---|---|
| **macro-F1** | Mean multi-label F1 across mood/topic/motif dims | higher |
| **macro-Jaccard** | Mean intersection-over-union across the same dims | higher |
| **mood_primary acc** | Exact-match accuracy on the single primary mood | higher |
| **intensity / access MAE** | Mean abs error on the 0-100 / 1-5 scalars | lower |
| **$/1k** | Cost to classify 1,000 poems at this model's rate | lower |
| **poems/min** | Throughput | higher |

Empty-vs-empty counts as a perfect match (the model correctly abstained), so
these metrics reward *not* hallucinating labels as much as finding real ones.

The printed `RECOMMENDATION:` line picks the **best macro-F1** candidate and, if a
different one is cheaper, names the cost trade so you can decide at 9k-poem scale.

In [ ]:
# !{cli("--bakeoff", "--candidates", *CANDIDATES, "--judge", JUDGE, \
#       "--batch-size", BATCH_SIZE, "--concurrency", CONCURRENCY)}

Load and display the bake-off results after a run (safe, no API):

In [ ]:
import json
if BAKEOFF_JSON.exists():
    bo = json.loads(BAKEOFF_JSON.read_text(encoding="utf-8"))
    print("Judge:", bo["gold"].get("judge_model"),
          "| gold scored:", bo["gold"].get("n_scored"),
          "| human-corrected:", bo["gold"].get("n_human"),
          "| calibration fired:", bo["gold"].get("calibration_fired"))
    print(f"\n{'model':34s} {'F1':>6s} {'Jacc':>6s} {'mood1':>6s} {'$/1k':>7s} {'p/min':>6s}")
    for r in bo["results"]:
        s = r["scores"]
        print(f"{r['model']:34s} {s['macro_f1']:6.3f} {s['macro_jaccard']:6.3f} "
              f"{s['mood_primary_accuracy']:6.3f} {r['cost_per_1k']:7.3f} {r['poems_per_min']:6.0f}")
    import re
    print("\nRECOMMENDATION:", re.sub(r"<[^>]+>", "", bo.get("recommendation", "")))
else:
    print("No bake-off file yet; run Step C first.")

## Step D: Interpret & decide (with reference numbers)

Use these numbers from the **last real run** to sanity-check a re-run. If your
re-run lands wildly off these, suspect a truncated run (empty-body 404s shrinking
coverage) or a taxonomy/version mismatch before trusting the result.

**Gold set:** 120 poems, judge `gemini-3.1-pro-preview`, **$0.68**, ~24 poems/min.

**Candidates vs judge:**

| Model | macro-F1 | Jaccard | mood_primary | $/1k | poems/min |
|---|---|---|---|---|---|
| `gemini/gemini-2.5-flash` | 0.738 | 0.634 | 0.708 | **$0.98** | 60 |
| `gemini/gemini-3.5-flash` | 0.785 | 0.688 | 0.658 | $3.88 | n/a |
| `gemini/gemini-3.6-flash` | **0.813** | **0.725** | **0.675** | $3.65 | n/a |

### Reading the trade-off

- **3.6-flash wins on quality** (F1 0.813, Jaccard 0.725) and is slightly cheaper
  than 3.5-flash.
- **2.5-flash is ~3.7x cheaper** ($0.98 vs $3.65 per 1k) and **fastest**
  (60 poems/min), at a real but bounded quality cost (~0.075 F1, ~0.09 Jaccard),
  and it actually scores **highest on `mood_primary`** (0.708).
- At ~9,072 poems the absolute cost gap is small (roughly $9 vs $33 for a full
  pass), so **quality can reasonably win** here, but if you re-bake regularly or
  scale the corpus, the cheap option's margin grows.

The `recommend()` function encodes exactly this: it names the best-F1 model and,
when a cheaper one exists, spells out the cost delta to weigh against the F1 gap.
Treat its output as the default, not the verdict: the `mood_primary` inversion
(cheap model best on the primary mood) is the kind of nuance worth a human eye.

> **Known gap: no per-poem predictions are saved.** The harness writes only
> **aggregate** metrics per candidate (`categorization_bakeoff.json` +
> `.html`), not each model's per-poem labels. So you can see *that* 3.6-flash
> scored 0.813, but you cannot open the run and inspect *which* poems it got
> wrong or how two candidates disagreed. A re-run is only aggregate-inspectable.
> **Suggested improvement:** add a `--dump-predictions` flag that writes each
> candidate's per-poem `{poem_id: labels}` (alongside the effective gold) to
> `data/categorization_predictions_<model_slug>.json`, so error analysis and
> confusion breakdowns are possible without re-labeling.

## Step E: Once you've chosen a model, bulk classify -> import -> backfill

**⚠️ This section writes to the production database.** It is the payoff after the
bake-off, but it is a *different* pipeline (`classify_poems.py` +
`import_categories.py`), included here as a pointer. Do not run it as part of a
bake-off.

### E1: Bulk classify the corpus

```bash
python -m poetry_quality_and_curation.categorization.classify_poems \
    --model gemini/gemini-3.6-flash \    # <- the model the bake-off picked
    --scope all \                         # all | top (--top-k N) | unclassified
    --concurrency 1 \                     # raise ONLY with real backoff
    --batch-size 4 \
    --max-cost 40 --resume
```

- `--scope unclassified` only labels poems with `categorized_at IS NULL` (safe to
  resume/top-up). `--scope top --top-k 5000` does the best-quality N first.
- Writes a Parquet checkpoint (`data/categories_<model_slug>.parquet`), not the
  DB. `--resume` skips poems already in that file. `--dry-run` prints stats and
  the dimension histogram without calling the API; use it first.

### E2: Import into the DB

```bash
python -m poetry_quality_and_curation.categorization.import_categories \
    --input poetry_quality_and_curation/categorization/data/categories_gemini_gemini-3.6-flash.parquet
```

Writes normalized `poem_categories` rows **and** denormalized scalars + JSONB
provenance on `poems`. Idempotent per poem (re-import replaces that poem's rows).
`--dry-run` shows the top labels per dimension without writing.

### E3: Backfill century from poet era (run once after import)

```bash
python -m poetry_quality_and_curation.categorization.import_categories --backfill-century
```

`century` is **never** model-guessed; it is derived deterministically from the
poet's era (`config.ERA_CENTURY`). No AI, no parquet, idempotent. Eras too broad
to pin are left `NULL` on purpose.

## Appendix: file map

Which files each step reads and writes (all under
`poetry_quality_and_curation/categorization/`):

| Step | Reads | Writes |
|---|---|---|
| A `--build-gold` | DB (`poems`, `poets`) | `data/categorization_gold.json` |
| B `--emit-review` | `data/categorization_gold.json` | `data/categorization_gold_review.json` |
| B `--apply-corrections FILE` | gold + edited review file | patches `categorization_gold.json` (adds `spot_check`) |
| C `--bakeoff` | `data/categorization_gold.json` | `data/categorization_bakeoff.json`, `data/categorization_bakeoff.html` |
| E1 `classify_poems` | DB (`poems`) | `data/categories_<model_slug>.parquet` |
| E2 `import_categories --input` | the parquet | DB: `poem_categories`, `poems` |
| E3 `import_categories --backfill-century` | DB (`poets.era_id`) | DB: `poems.century` |

### Source of truth

- Harness / modes / flags: `eval_categorization.py` (docstring + `parse_args`).
- Bulk classifier: `classify_poems.py`. Importer: `import_categories.py`.
- Taxonomy, provider routing, DB connection, `ERA_CENTURY`: `config.py`.

### One-line reminders

- **Concurrency = 1** on every API step until you add backoff.
- **Empty-body 404 = rate limit** (slow down); **404-with-body = retired model**
  (swap it out).
- Reference: gold 120 poems / $0.68; best candidate `3.6-flash` F1 0.813; cheapest
  `2.5-flash` $0.98/1k.
- Bake-off saves aggregate metrics only, not per-poem predictions (see Step D).
- Step E writes to **production**; keep it out of bake-off runs.